# LRM51 transcriptomic predictor for PD-1/PD-L1 immunotherapy response

This notebook prepares the expression matrix, applies CPM normalization, evaluates a logistic regression model, interprets the selected genes with SHAP, and summarizes the final 51-gene model performance.

> **Note:** Update the file paths in the data-loading section before running the notebook outside the original Google Colab environment.


### Reproducibility notes

- Keep all random seeds fixed when reproducing model performance.
- Apply oversampling only within the training data.
- Do not recompute normalization parameters using validation or test labels.


## 1. Environment setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.feature_selection as fs
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.model_selection import train_test_split
import sklearn.metrics as metrics
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn import cluster, datasets, mixture
from sklearn.cluster import KMeans
from itertools import cycle, islice
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import cross_val_score
from sklearn.metrics import recall_score
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from statistics import mean, stdev
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import (
    RidgeClassifier, SGDClassifier, PassiveAggressiveClassifier, Perceptron
)
from sklearn.svm import SVC, NuSVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB, MultinomialNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from lightgbm import LGBMClassifier
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.stats import zscore
from imblearn.over_sampling import SMOTE, ADASYN
from collections import Counter
from sklearn.manifold import TSNE
import itertools
from sklearn.metrics import roc_auc_score, f1_score


In [ ]:
import sys
import sklearn
import imblearn
import shap
import pandas as pd
import numpy as np
print("Python:", sys.version)
print("scikit-learn:", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("shap:", shap.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

In [ ]:
!pip install shap
import shap

In [ ]:
!pip install scikit-posthocs
import scikit_posthocs as sp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import matplotlib.font_manager as fm

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13
})

## 2. Load count matrices and labels

In [ ]:
# Raw count matrix
raw_counts = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AUNA/crudaCompleta.csv", header=0, index_col=0, sep=",")
raw_counts = raw_counts.dropna()
raw_counts = raw_counts.astype("float")
raw_counts = raw_counts.transpose()
raw_counts

In [ ]:
raw_counts = raw_counts.drop(index=['ERR2498029', 'EGAR00001766803']) # Remove samples with artifacts in the gene-read density curve
raw_counts

In [ ]:
# Load the first clinical metadata table
clinical_metadata = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AUNA/ListaPacientes2912023-3.csv")
clinical_metadata = clinical_metadata[clinical_metadata['run_accession'] != 'EGAR00001766803']
y = clinical_metadata['Best RECIST response']
y = np.array([0 if (value.strip() == 'PD' or value.strip() == 'SD' or value.strip() == 'NR') else 1 for value in y])
clinical_metadata['Best RECIST response'] = y
clinical_metadata['Best RECIST response'].value_counts()
clinical_metadata

In [ ]:
# Load the second clinical metadata table
clinical_metadata_val = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AUNA/listaPacientesVal.csv")
# Remove the sample flagged as an artifact
clinical_metadata_val = clinical_metadata_val[clinical_metadata_val['run_accession'] != 'ERR2498029']
y2 = clinical_metadata_val['Best RECIST response']
y2 = np.array([0 if (value.strip() == 'PD' or value.strip() == 'SD' or value.strip() == 'NR') else 1 for value in y2])
clinical_metadata_val['Best RECIST response'] = y2
clinical_metadata_val['Best RECIST response'].value_counts()
clinical_metadata_val

In [ ]:
clinical_metadata_val["Tipo"] = clinical_metadata_val["Base"].apply(lambda x: "lung" if x == "CHO" else "Gastrico" if x == "KIM2" else "Melanoma")
clinical_metadata_val

In [ ]:
# Merge both clinical metadata tables
clinical_metadata = pd.concat([clinical_metadata, clinical_metadata_val])
clinical_metadata

In [ ]:
# Merge response labels
y = np.concatenate((y, y2))
y

In [ ]:
import matplotlib.pyplot as plt

# ==========================
# Publication-quality plotting settings
# ==========================
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,          # Keep labels readable for publication figures
    'axes.titlesize': 16,
    'axes.titleweight': 'bold'
})

# Crear un diccionario para mapear valores binarios a etiquetas
label_mapping = {
    0: 'No Responser',
    1: 'Responser'
}

# Contar valores
value_counts = clinical_metadata['Best RECIST response'].value_counts()

# Reemplazar números por etiquetas
labels = [label_mapping[val] for val in value_counts.index]

# Create figure (más grande para publicación)
fig, ax = plt.subplots(figsize=(7,7))

# Pie chart
wedges, texts, autotexts = ax.pie(
    value_counts,
    labels=labels,
    autopct='%.1f%%',
    startangle=90,
    counterclock=False,
    labeldistance=1.08,
    pctdistance=0.65,
    wedgeprops=dict(
        edgecolor='white',
        linewidth=2
    ),
    textprops=dict(
        fontsize=14,
        fontweight='bold'
    )
)

# Formato de los porcentajes
for autotext in autotexts:
    autotext.set_fontsize(14)
    autotext.set_fontweight('bold')
    autotext.set_color('black')

# Formato de las etiquetas
for text in texts:
    text.set_fontsize(14)
    text.set_fontweight('bold')

# Título
ax.set_title(
    "RECIST-Based Response Distribution",
    fontsize=16,
    fontweight='bold',
    pad=20
)

# Mantener forma circular
ax.axis('equal')

# Save figure en TIFF 600 dpi
plt.savefig(
    "RECIST_Response_Distribution.tiff",
    format="tiff",
    dpi=600,
    bbox_inches="tight",
    pil_kwargs={"compression": "raw"}
)

plt.show()

In [ ]:
np.array_equal(raw_counts.index, clinical_metadata['run_accession']	)

In [ ]:
# Replace slashes in sample identifiers to match downstream tables
raw_counts.index = raw_counts.index.str.replace('/', '.')
raw_counts

In [ ]:

# Harmonize cancer type labels
cancer_type_corrections = {
    "thymic": "Thymic",
    "Gastrico": "Gastric",
    "lung": "Lung",
    "oral": "Oral",
    "Urothelial": "Urothelial",
    # Agrega aquí otras cancer_type_corrections necesarias
}

# Aplicar las cancer_type_corrections
clinical_metadata["Tipo"] = clinical_metadata["Tipo"].replace(cancer_type_corrections)

In [ ]:
import matplotlib.pyplot as plt

# ==========================
# Publication-quality settings
# ==========================
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13
})

# Count patients by cancer type (ordered from highest to lowest frequency)
value_counts = clinical_metadata['Tipo'].value_counts().sort_values(ascending=False)

# Calculate percentages
percentages = value_counts / value_counts.sum() * 100

# Create figure
fig, ax = plt.subplots(figsize=(9, 6))

# Horizontal bar plot
bars = ax.barh(
    value_counts.index,
    value_counts.values,
    edgecolor='black',
    linewidth=1.0
)

# Highest frequency at the top
ax.invert_yaxis()

# Add sample size and percentage
for i, bar in enumerate(bars):
    width = bar.get_width()
    ax.text(
        width + 0.3,
        bar.get_y() + bar.get_height()/2,
        f'n={int(width)} ({percentages.iloc[i]:.1f}%)',
        va='center',
        ha='left',
        fontsize=13
    )

# Labels and title
ax.set_xlabel("Number of Patients")
ax.set_ylabel("Cancer Type")
ax.set_title(
    "Distribution of Patients by Cancer Type",
    pad=18
)

# Add some extra space for labels
ax.set_xlim(0, value_counts.max() * 1.30)

# Light grid
ax.grid(
    axis='x',
    linestyle='--',
    linewidth=0.6,
    alpha=0.4
)

# Remove unnecessary borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Optimize layout
plt.tight_layout()

# Save as TIFF (600 dpi)
plt.savefig(
    "Cancer_Type_Distribution.tiff",
    format="tiff",
    dpi=600,
    bbox_inches="tight",
    pil_kwargs={"compression": "raw"}
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ==========================
# Publication-quality settings
# ==========================
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
    'legend.title_fontsize': 12
})

# Ensure response is numeric
clinical_metadata['Best RECIST response'] = clinical_metadata['Best RECIST response'].astype(int)

# Count patients by cancer type and response
conteo = (
clinical_metadata
    .groupby(['Tipo', 'Best RECIST response'])
    .size()
    .unstack(fill_value=0)
)

# Ensure both categories exist
for col in [0, 1]:
    if col not in conteo.columns:
        conteo[col] = 0

conteo = conteo[[0, 1]]

# Sort by total number of patients
conteo = conteo.loc[
    conteo.sum(axis=1).sort_values(ascending=False).index
]

# Percentages within each cancer type
porcentajes = conteo.div(conteo.sum(axis=1), axis=0) * 100

# Create figure
fig, ax = plt.subplots(figsize=(10, 6))

# Sobriety colors
colors = ['#B0B0B0', '#4C72B0']

# Plot
conteo.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    color=colors,
    edgecolor='black',
    linewidth=0.8
)

# Add percentage labels
for i, cancer_type in enumerate(conteo.index):
    cumulative = 0

    for response in [0, 1]:
        count = conteo.loc[cancer_type, response]
        pct = porcentajes.loc[cancer_type, response]

        if count > 0:

            if count >= 5:
                # Label inside large segments
                ax.text(
                    i,
                    cumulative + count / 2,
                    f'{pct:.1f}%',
                    ha='center',
                    va='center',
                    fontsize=12
                )
            else:
                # Label outside small segments with horizontal offset
                x_offset = -0.18 if response == 0 else 0.18
                y_offset = 0.30 if response == 0 else 0.80

                ax.text(
                    i + x_offset,
                    cumulative + count + y_offset,
                    f'{pct:.1f}%',
                    ha='center',
                    va='bottom',
                    fontsize=12
                )

        cumulative += count

# Labels and title
ax.set_xlabel("Cancer Type")
ax.set_ylabel("Number of Patients")
ax.set_title("Distribution of Responsers and Non-Responsers by Cancer Type")

# Legend
ax.legend(
    ["Non-responder", "Responser"],
    title="RECIST Response",
    frameon=False,
    loc='upper right'
)

# X-axis labels
plt.xticks(rotation=45, ha='right')

# Grid
ax.grid(
    axis='y',
    linestyle='--',
    linewidth=0.6,
    alpha=0.4
)

# Remove borders
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# More vertical room
ax.set_ylim(0, conteo.sum(axis=1).max() * 1.25)

plt.tight_layout()

# Save as TIFF
plt.savefig(
    "Responsers_by_Cancer_Type.tiff",
    format="tiff",
    dpi=600,
    bbox_inches="tight",
    pil_kwargs={"compression": "raw"}
)

plt.show()

## 3. Define the 51-gene panel

In [ ]:
selected_genes =['SFTPC',
 'CSRP3',
 'LINC00431',
 'MAP4K1',
 'MAGEA10',
 'CXCL10',
 'PKD1L2',
 'SLC6A12',
 'AATF',
 'PGBD2',
 'EPHA8',
 'TXLNGY',
 'LOC124904611',
 'PDIK1L',
 'PTX4',
 'LINC00518',
 'JAKMIP1',
 'SH2D5',
 'REXO5',
 'LOC124903104',
 'MIR604',
 'PAGE5',
 'QRSL1',
 'EOMES',
 'LOC124900759',
 'CYP2S1',
 'CCDC140',
 'LOC124909366',
 'LOC124900511',
 'RWDD2B',
 'LOC124906091',
 'KCNC2',
 'LOC102724334',
 'HOXD1',
 'LIF',
 'RTP5',
 'LOC642131',
 'LOC107985745',
 'ARHGEF35-AS1',
 'KASH5',
 'LOC124901738',
 'DPYSL5',
 'GML',
 'FAM180A',
 'PSME1',
 'ITGAE',
 'LOC101926984',
 'SCG2',
 'RNU6-9',
 'RANGRF',
 'LOC101927723',
 'LRFN4',
 'TLX1',
 'LMBRD2',
 'LOC105372273',
 'LOC644277',
 'MIR542',
 'MEST',
 'CREB3L3',
 'NEUROG2',
 'GRB10',
 'ANKRD18DP',
 'RDM1',
 'TRMT1L',
 'LOC124904912',
 'NEUROG2-AS1',
 'MUC15',
 'LOC105372521',
 'MYADML2',
 'LOC124908054',
 'KLHL29',
 'LOC105370964',
 'LOC105374618',
 'EGR1',
 'RIMBP3B',
 'TRBV11-3',
 'AMN1',
 'GPATCH1', 'AATF',     'LCMT1',    'TXLNGY',
        'GALNT1',  'CEP89', 'LMBRD2', 'GRB10', 'PARP14','AKR1E2', 'PRSS12', 'ANKRD27', 'NSMF',
        'EDC3', 'MAGEA10', 'PAX3', 'MAP4K1', 'COL17A1', 'UBALD2',
       'KLHL29',  'ZSCAN2', 'NACAD', 'LOC102724159',
       'TRMT1L', 'DPYSL5',  'BRF2', 'AP1AR', 'ST8SIA6', 'SIM1',
       'MALL',  'ITGAE',  'MEST', 'HOXC10', 'COL23A1',
       'RDH11', 'RNASEH2A', 'PKD1L2', 'B4GALNT3', 'NR1D2',
      'SLC34A2', 'TVP23B', 'KLRC1', 'SST','PITPNM2', 'BIRC2', 'SLIT2',
       'ARL4C', 'CXCL9', 'CTAGE15', 'ZNF649', 'CPNE8',
       'SFTPC', 'FAT4', 'PM20D1', 'PERP',  'PAPLN', 'SLC6A12',
       'BICC1',  'NHS' , 'CSRP3']


In [ ]:
#dejar elementos únicos en selected_genes
selected_genes = list(set(selected_genes))
#numero de elementos en selected_genes
len(selected_genes)

In [ ]:
expression_matrix = raw_counts.copy()
expression_matrix

## 4. CPM normalization

In [ ]:
# Convert raw counts to CPM
library_sizes = np.array([np.sum(expression_matrix.values, axis=1)])
for i in range(library_sizes.shape[1]):
  expression_matrix.iloc[i,:] = expression_matrix.iloc[i,:] * (1000 / library_sizes[0,i])
expression_matrix


In [ ]:
expression_matrix =expression_matrix*1000
expression_matrix

In [ ]:
expression_matrix

In [ ]:
expression_matrix =np.log2(expression_matrix+1)
expression_matrix

## 5. Gene filtering

In [ ]:
expression_matrix =expression_matrix[selected_genes]
expression_matrix

## 6. Train/validation/test split and SMOTE

The model uses predefined sample splits to preserve the original experimental design. SMOTE is applied only to the training set to avoid information leakage.

- Validation set: 5%
- Test set: 10%


In [ ]:
# Load predefined train, validation, and test splits
X_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Tesis/conjuntos/genes_train.csv", header=0, index_col=0, sep=",")
y_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Tesis/conjuntos/label_train.csv", header=0, index_col=0, sep=",")
X_val = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Tesis/conjuntos/genes_val.csv", header=0, index_col=0, sep=",")
y_val = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Tesis/conjuntos/label_val.csv", header=0, index_col=0, sep=",")
X_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Tesis/conjuntos/genes_test.csv", header=0, index_col=0, sep=",")
y_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Tesis/conjuntos/label_test.csv", header=0, index_col=0, sep=",")


In [ ]:
X_valt =expression_matrix[expression_matrix.index.isin(X_val.iloc[:,0])]
X_testt =expression_matrix[expression_matrix.index.isin(X_test.iloc[:,0])]
X_traint =expression_matrix[expression_matrix.index.isin(X_train.iloc[:,0])]

In [ ]:
X_val = X_valt.reindex(X_val.iloc[:,0])
X_test = X_testt.reindex(X_test.iloc[:,0])
X_train = X_traint.reindex(X_train.iloc[:,0])

In [ ]:
X_cruz = pd.concat([X_train, X_val], axis=0)
X_cruz


In [ ]:
# Convert label raw_counts frames to NumPy arrays
y_train = y_train.iloc[:,0].to_numpy()
y_val = y_val.iloc[:,0].to_numpy()
y_test = y_test.iloc[:,0].to_numpy()
y_val

In [ ]:
y_cruz = np.concatenate((y_train, y_val), axis=0)
y_cruz

In [ ]:
# Apply SMOTE only to the training set
X_traine, y_traine = SMOTE(random_state=68).fit_resample(X_train, y_train)
print(sorted(Counter(y_traine).items()))

## 7. SFTPC descriptive analysis

In [ ]:
clinical_metadata = clinical_metadata.set_index(clinical_metadata.columns[0])
clinical_metadata

In [ ]:
train_metadata = pd.concat([X_train, clinical_metadata.loc[X_train.index]], axis=1)

In [ ]:
train_metadata

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

graficar = (
    train_metadata
    .sort_values("SFTPC", ascending=False)
    .head(100)
    .copy()
)
colors = graficar["Best RECIST response"].map({
    0: "red",
    1: "blue"
})

plt.figure(figsize=(12,4))

plt.bar(
    np.arange(len(graficar)),
    graficar["SFTPC"],
    color=colors
)

plt.xlabel("Samples (ordered by SFTPC)")
plt.ylabel("SFTPC expression")
plt.title("SFTPC expression ordered from highest to lowest")


plt.tight_layout()

# Save figure para artículo
plt.savefig(
    "SFTPC_ordered_expressionComplete.png",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "SFTPC_ordered_expressionComplete.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
expression_matrix = (
    train_metadata
    .sort_values("SFTPC", ascending=False)
    .head(80)
    .copy()
)
plt.figure(figsize=(12,4))

plt.scatter(
    range(len(expression_matrix)),
    expression_matrix["SFTPC"],
    c=expression_matrix["Best RECIST response"],
    s=50
)

plt.xlabel("Samples ordered by SFTPC")
plt.ylabel("SFTPC expression")

plt.show()

In [ ]:
display(
    train_metadata
    .sort_values("SFTPC", ascending=False)
    .head(60)[["SFTPC", "Best RECIST response", "Tipo"]]
)

In [ ]:
selected_cancer_type = "Renal"   # ejemplo

filtered_metadata = (
    train_metadata[train_metadata["Tipo"] == selected_cancer_type]
    .sort_values("SFTPC", ascending=False)
)

filtered_metadata[["Tipo", "SFTPC", "Best RECIST response"]]

In [ ]:
results = []

for cancer_type, group in train_metadata.groupby("Tipo"):

    ordered_group = group.sort_values("SFTPC", ascending=False)
    resp = ordered_group["Best RECIST response"].values

    transitions = []
    for i in range(len(resp) - 1):
        if resp[i] == 0 and resp[i + 1] == 1:
            transitions.append(i)

    if len(transitions) > 0:
        idx = transitions[0]   # primer cambio 0 → 1
        row = ordered_group.iloc[idx]

        results.append({
            "Tipo": cancer_type,
            "SFTPC_cutoff_first_0": row["SFTPC"],
            "Sample_first_0": row.name,
            "Best_RECIST_response": row["Best RECIST response"]
        })
    else:
        results.append({
            "Tipo": cancer_type,
            "SFTPC_cutoff_first_0": None,
            "Sample_first_0": None,
            "Best_RECIST_response": None
        })

matriz_resultados = pd.DataFrame(results)
display(matriz_resultados)

In [ ]:
results = []

for cancer_type, group in train_metadata.groupby("Tipo"):

    ordered_group = group.sort_values("SFTPC", ascending=False)

    last_valid_zero = None

    for idx, row in ordered_group.iterrows():

        response = row["Best RECIST response"]
        sftpc = row["SFTPC"]

        if response == 0 and sftpc > 3:
            last_valid_zero = (idx, row)

        elif response == 1:

            if last_valid_zero is not None:
                sample_idx, fila_corte = last_valid_zero

                results.append({
                    "Tipo": cancer_type,
                    "SFTPC_cutoff": fila_corte["SFTPC"],
                    "Sample_cutoff": sample_idx
                })

            else:
                results.append({
                    "Tipo": cancer_type,
                    "SFTPC_cutoff": 3,
                    "Sample_cutoff": None
                })

            break

    # Caso en que nunca apareció un 1
    else:
        if last_valid_zero is not None:
            sample_idx, fila_corte = last_valid_zero

            results.append({
                "Tipo": cancer_type,
                "SFTPC_cutoff": fila_corte["SFTPC"],
                "Sample_cutoff": sample_idx
            })
        else:
            results.append({
                "Tipo": cancer_type,
                "SFTPC_cutoff": 3,
                "Sample_cutoff": None
            })

matriz_resultados = pd.DataFrame(results)

display(matriz_resultados)

In [ ]:
for cancer_type, group in train_metadata.groupby("Tipo"):
    print("\n", cancer_type)
    display(
        group.sort_values("SFTPC", ascending=False)[
            ["SFTPC", "Best RECIST response"]
        ]
    )

## 8. Baseline model evaluation

In [ ]:
baseline_model =LogisticRegression(C=0.1, solver= 'liblinear') # Alternative solver options
#baseline_model=AdaBoostClassifier(learning_rate= 1, n_estimators= 100) # Alternative solver options
#baseline_model=GradientBoostingClassifier( n_estimators= 100, criterion="friedman_mse") # Alternative solver options
#baseline_model=AdaBoostClassifier(learning_rate= 1, n_estimators= 100) # Alternative solver options
#baseline_model=LGBMClassifier(learning_rate= 0.2, n_estimators= 200, num_leaves=31)
#baseline_model=PassiveAggressiveClassifier(C= 0.1)
#baseline_model=Perceptron(alpha= 0.0001, penalty= 'l2')
baseline_model.fit(X_traine, y_traine)
y_pred =baseline_model.predict(X_test)
print(classification_report(y_test, y_pred))
cm =confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No response", "Response"])
disp.plot(cmap=plt.cm.Blues)

In [ ]:
baseline_model.fit(X_traine, y_traine)
y_pred =baseline_model.predict(X_val)
print(classification_report(y_val, y_pred))
cm =confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No response", "Response"])
disp.plot(cmap=plt.cm.Blues)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
# Estimate probabilities for the positive class
y_probabilities = baseline_model.predict_proba(X_val)[:, 1]

# Compute AUC
auc = roc_auc_score(y_val, y_probabilities)
print(f"AUC: {auc:.4f}")

# Graficar curva ROC
fpr, tpr, _ = roc_curve(y_val, y_probabilities)
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")  # Línea de azar
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC - Logistic Regression")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score
auc_scores = baseline_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, auc_scores)
auc

In [ ]:
# Estimate decision scores instead of probabilities
decision_scores = baseline_model.decision_function(X_test)

# Compute AUC
auc = roc_auc_score(y_test, decision_scores)
print(f"AUC: {auc:.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score
auc_scores = baseline_model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, auc_scores)
auc

In [ ]:
# Estimate decision scores instead of probabilities
decision_scores = baseline_model.decision_function(X_val)

# Compute AUC
auc = roc_auc_score(y_val, decision_scores)
print(f"AUC: {auc:.4f}")

In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, balanced_accuracy_score, f1_score
)

baseline_model.fit(X_traine, y_traine)

# 2) Estimate probabilities for the positive class
y_prob = baseline_model.predict_proba(X_test)[:, 1]

# 3) Buscar mejor threshold (elige una métrica)
thresholds = np.linspace(0, 1, 1001)

best_t = 0.5
best_score = -np.inf

metric = "youden"  # "f1" o "bal_acc" o "youden"

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)

    if metric == "f1":
        score = f1_score(y_test, y_pred, zero_division=0)
    elif metric == "bal_acc":
        score = balanced_accuracy_score(y_test, y_pred)
    elif metric == "youden":
        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-12)
        spec = tn / (tn + fp + 1e-12)
        score = sens + spec - 1.0
    else:
        raise ValueError("metric debe ser: 'f1', 'bal_acc' o 'youden'")

    if score > best_score:
        best_score = score
        best_t = float(t)

print(f"✅ Mejor threshold por {metric}: t={best_t:.4f} | score={best_score:.4f}")

best_t =0.4

# 4) Final prediction using the optimal threshold
y_pred_best = (y_prob >= best_t).astype(int)

# Optional: AUC is threshold-independent
auc = roc_auc_score(y_test, y_prob)
print(f"📈 AUC (threshold-free): {auc:.4f}")

# 5) Reporte y matriz de confusión
print(classification_report(y_test, y_pred_best, target_names=["No response", "Response"]))

cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No response", "Response"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix (t={best_t:.3f})")
plt.show()


In [ ]:
baseline_model.fit(X_traine, y_traine)

# 2) Estimate probabilities for the positive class
y_prob = baseline_model.predict_proba(X_val)[:, 1]

# 4) Final prediction using the optimal threshold
y_pred_best = (y_prob >= best_t).astype(int)

# Optional: AUC is threshold-independent
auc = roc_auc_score(y_val, y_prob)
print(f"📈 AUC (threshold-free): {auc:.4f}")

# 5) Reporte y matriz de confusión
print(classification_report(y_val, y_pred_best, target_names=["No response", "Response"]))

cm = confusion_matrix(y_val, y_pred_best, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No response", "Response"])
disp.plot(cmap=plt.cm.Blues)
plt.title(f"Confusion Matrix (t={best_t:.3f})")
plt.show()


In [ ]:
# Gradient Boosting
#baseline_model = GradientBoostingClassifier(n_estimators=100, criterion= 'friedman_mse' )
#baseline_model = DecisionTreeClassifier (criterion = 'entropy', max_depth= 20 )
y_pred =baseline_model.predict(expression_matrix)
print(classification_report(y, y_pred))
cm =confusion_matrix(y, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No response", "Response"])
disp.plot(cmap=plt.cm.Blues)

In [ ]:
# Estimate decision scores instead of probabilities
decision_scores = baseline_model.decision_function(expression_matrix)

# Compute AUC
auc = roc_auc_score(y, decision_scores)
print(f"AUC: {auc:.4f}")

In [ ]:
# Replace slashes in clinical sample identifiers
clinical_metadata["run_accession"] = clinical_metadata["run_accession"].str.replace("/", ".")


## 9. SHAP model interpretation

In [ ]:
# SHAP explainer for the linear model
#explainer = shap.Explainer(baseline_model)
explainer = shap.Explainer(baseline_model, X_test)
shap_values = explainer(X_test)

# Display SHAP values
shap.summary_plot(shap_values, X_test, max_display=70)

In [ ]:
shap.plots.bar(shap_values,  max_display=70)

In [ ]:
vals = np.abs(shap_values.values).mean(0)
feature_names = X_train.columns
feature_importance = pd.DataFrame(list(zip(feature_names, vals)),
                                 columns=['col_name','feature_importance_vals'])
feature_importance.sort_values(by=['feature_importance_vals'],
                              ascending=False, inplace=True)


# Definir el threshold mínimo de importancia (ajústalo según tu caso)
threshold = 0.00  # Cambia este valor según la distribución de importancia

# Filtrar solo las features con importancia mayor al threshold
shap_feature_ranking = feature_importance[feature_importance["feature_importance_vals"] > threshold].copy()

# Convertir en DataFrame y acumular en selected_genes
#shap_feature_ranking = pd.DataFrame(feature_importance)
#shap_feature_ranking = pd.DataFrame(feature_importance.iloc[0:60, :])  # Seleccionar las 60 más importantes

shap_feature_ranking.shape

In [ ]:
shap_feature_ranking

In [ ]:
#ordenar shap_feature_ranking de menor a mayor por la columna feature_importance_vals
shap_feature_ranking = shap_feature_ranking.sort_values(by="feature_importance_vals", ascending=False)
shap_feature_ranking


In [ ]:
# Convert the selected features to a NumPy array
shap_feature_ranking["col_name"].to_numpy()

In [ ]:
shap_selected_features = shap_feature_ranking["col_name"].to_numpy()
shap_selected_features

## 10. Cross-validation

In [ ]:
import itertools
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_score, recall_score, f1_score
)

from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
import shap
import matplotlib.pyplot as plt

strtfdKFold = StratifiedKFold(n_splits=10, shuffle=True, random_state=5)
kfold = strtfdKFold.split(X_train, y_train)
scores = []
roc_score = []
f1_scores = []
precision_scores = []
recall_scores = []
specificity_scores = []
predicted_labels = []
observed_labels = []
i = 0
THRESHOLD = 0.5   # ejemplo

# Inicializar DataFrame para acumular shap_feature_ranking
selected_genes = pd.DataFrame()

for train_index, test_index in kfold:
    baseline_model = LogisticRegression(C= 0.1, solver= 'liblinear')
    X_train_cv, X_test_cv = X_train.values[train_index], X_train.values[test_index]
    y_train_cv, y_test_cv = y_train[train_index], y_train[test_index]

    # oversampling para el modelo
    X_train_cv, y_train_cv = SMOTE().fit_resample(X_train_cv, y_train_cv)

    baseline_model.fit(X_train_cv, y_train_cv)
    scores.append(baseline_model.score(X_test_cv, y_test_cv))

    auc_scores = baseline_model.predict_proba(X_test_cv)[:, 1]
    roc_score.append(roc_auc_score(y_test_cv, auc_scores))

    y_prob = baseline_model.predict_proba(X_test_cv)[:, 1]
    py = (y_prob >= THRESHOLD).astype(int)
    predicted_labels = list(itertools.chain(predicted_labels, py.flatten().tolist()))
    observed_labels = list(itertools.chain(observed_labels, y_test_cv.flatten().tolist()))

    # Compute métricas fold a fold

    f1_scores.append(f1_score(y_test_cv, py, average="macro"))
    precision_scores.append(precision_score(y_test_cv, py,average="macro"))
    recall_scores.append(recall_score(y_test_cv, py))

    # Compute especificidad (specificity)
    cm = confusion_matrix(y_test_cv, py)
    if cm.shape == (2, 2):  # Para evitar errores si solo hay una clase en el fold
        TN = cm[0, 0]
        FP = cm[0, 1]
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
        specificity_scores.append(specificity)
    else:
        specificity_scores.append(np.nan)

    print('Resultado iteración:', i)
    print(train_index)
    print(classification_report(y_test_cv, py))
    cm_val = confusion_matrix(y_test_cv, py)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_val, display_labels=["No response", "Response"])
    disp.plot(cmap=plt.cm.Greens)
    i += 1

# Cálculo de promedios y desviaciones estándar
accuracy_test = np.mean(scores)
accuracy_desv = np.std(scores)

auc_test = np.mean(roc_score)
auc_test_desv = np.std(roc_score)

f1_test = np.mean(f1_scores)
f1_test_desv = np.std(f1_scores)

precision_test = np.mean(precision_scores)
precision_test_desv = np.std(precision_scores)

recall_test = np.mean(recall_scores)
recall_test_desv = np.std(recall_scores)

specificity_test = np.nanmean(specificity_scores)
specificity_test_desv = np.nanstd(specificity_scores)

print("\nFinal cross-validation results (mean ± SD):")
print(f"Accuracy:     {accuracy_test:.3f} ± {accuracy_desv:.3f}")
print(f"AUC:          {auc_test:.3f} ± {auc_test_desv:.3f}")
print(f"F1-score:     {f1_test:.3f} ± {f1_test_desv:.3f}")
print(f"Precision:    {precision_test:.3f} ± {precision_test_desv:.3f}")
print(f"Recall:       {recall_test:.3f} ± {recall_test_desv:.3f}")
print(f"Specificity:  {specificity_test:.3f} ± {specificity_test_desv:.3f}")


In [ ]:
print("\nFinal cross-validation results (mean ± SD):")
print(f"Accuracy:     {accuracy_test:.3f} ± {accuracy_desv:.3f}")
print(f"AUC:          {auc_test:.3f} ± {auc_test_desv:.3f}")
print(f"F1-score:     {f1_test:.3f} ± {f1_test_desv:.3f}")
print(f"Precision:    {precision_test:.3f} ± {precision_test_desv:.3f}")
print(f"Recall:       {recall_test:.3f} ± {recall_test_desv:.3f}")
print(f"Specificity:  {specificity_test:.3f} ± {specificity_test_desv:.3f}")

In [ ]:
print(classification_report(observed_labels, predicted_labels))

In [ ]:
cm =[confusion_matrix(observed_labels,predicted_labels)]
disp = ConfusionMatrixDisplay(confusion_matrix=cm[0], display_labels=["No response", "Response"])
disp.plot(cmap=plt.cm.Blues)

## 11. Final 51-gene model

In [ ]:
X_traine2 = X_traine[shap_selected_features]
X_train2 = X_train[shap_selected_features]
X_test2 = X_test[shap_selected_features]
X_val2 = X_val[shap_selected_features]
X_cruz2 = X_cruz[shap_selected_features]
X_traine2

In [ ]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.15)  # adjust the threshold according to your dataset
filtered_feature_matrix = selector.fit_transform(X_train2)
# Retrieve the retained columns
selected_columns = X_train2.columns[selector.get_support()]
filtered_feature_matrix = pd.DataFrame(filtered_feature_matrix, columns=selected_columns)


In [ ]:
low_variance_features = X_train2.columns[~selector.get_support()]
print("Features removed by low variance:", list(low_variance_features))

In [ ]:
variance_selected_features =filtered_feature_matrix.columns.to_numpy()
variance_selected_features

In [ ]:
X_traine2 = X_traine[variance_selected_features]
X_train2 = X_train[variance_selected_features]
X_test2 = X_test[variance_selected_features]
X_val2 = X_val[variance_selected_features]
X_cruz2 = X_cruz[variance_selected_features]
X_traine2

In [ ]:

# X: feature matrix
# y: binary target vector

candidate_features = list(X_traine2.columns)

# Function to compute mean AUC using cross-validation
def mean_auc(X_subset, y, cv_folds=5):
    model = LogisticRegression(C= 0.1, solver= 'liblinear')
    aucs = cross_val_score(model, X_subset, y, cv=cv_folds, scoring='roc_auc')
    return aucs.mean()

# AUC inicial con todas las candidate_features
best_auc = mean_auc(X_traine2[candidate_features], y_traine)

# Save figure el registro del proceso
registro = []

# Iteración secuencial
i = 0
while i < len(candidate_features):
    var_actual = candidate_features[i]
    vars_sin_actual = candidate_features[:i] + candidate_features[i+1:]

    auc_tmp = mean_auc(X_traine2[vars_sin_actual], y_traine)

    # Decisión: feature_to_remove si mejora o se mantiene el AUC
    if auc_tmp >= best_auc:
        eliminado = True
        candidate_features = vars_sin_actual
        best_auc = auc_tmp
        # no avanzar i, porque los índices cambiaron
    else:
        eliminado = False
        i += 1  # pasar a la siguiente variable

    registro.append({
        'variable': var_actual,
        'auc_nuevo': auc_tmp,
        'eliminado': eliminado,
        'auc_base': best_auc
    })

# Convertir results a DataFrame
feature_elimination_results = pd.DataFrame(registro)



In [ ]:
feature_elimination_results

In [ ]:
len(candidate_features)

In [ ]:
X_traine2 = X_traine[candidate_features]
X_train2 = X_train[candidate_features]
X_test2 = X_test[candidate_features]
X_val2 = X_val[candidate_features]
X_cruz2 = X_cruz[candidate_features]
X_traine2

In [ ]:

# X: feature matrix
# y: binary target vector

candidate_features = list(X_traine2.columns)

# Function to compute mean AUC using cross-validation
def mean_auc(X_subset, y, cv_folds=5):
    model = LogisticRegression(C= 0.1, solver= 'liblinear')
    aucs = cross_val_score(model, X_subset, y, cv=cv_folds, scoring='roc_auc')
    return aucs.mean()

# AUC inicial con todas las candidate_features
best_auc = mean_auc(X_traine2[candidate_features], y_traine)

# Save figure el registro del proceso
registro = []

# Iteración secuencial
i = 0
while i < len(candidate_features):
    var_actual = candidate_features[i]
    vars_sin_actual = candidate_features[:i] + candidate_features[i+1:]

    auc_tmp = mean_auc(X_traine2[vars_sin_actual], y_traine)

    # Decisión: feature_to_remove si mejora o se mantiene el AUC
    if auc_tmp >= best_auc:
        eliminado = True
        candidate_features = vars_sin_actual
        best_auc = auc_tmp
        # no avanzar i, porque los índices cambiaron
    else:
        eliminado = False
        i += 1  # pasar a la siguiente variable

    registro.append({
        'variable': var_actual,
        'auc_nuevo': auc_tmp,
        'eliminado': eliminado,
        'auc_base': best_auc
    })

# Convertir results a DataFrame
feature_elimination_results = pd.DataFrame(registro)



In [ ]:
len(candidate_features)

In [ ]:
X_traine2 = X_traine[candidate_features]
X_train2 = X_train[candidate_features]
X_test2 = X_test[candidate_features]
X_val2 = X_val[candidate_features]
X_cruz2 = X_cruz[candidate_features]
X_traine2

In [ ]:

# X: feature matrix
# y: binary target vector

candidate_features = list(X_traine2.columns)

# Function to compute mean AUC using cross-validation
def mean_auc(X_subset, y, cv_folds=5):
    model = LogisticRegression(C= 0.1, solver= 'liblinear')
    aucs = cross_val_score(model, X_subset, y, cv=cv_folds, scoring='roc_auc')
    return aucs.mean()

# AUC inicial con todas las candidate_features
best_auc = mean_auc(X_traine2[candidate_features], y_traine)

# Save figure el registro del proceso
registro = []

# Iteración secuencial
i = 0
while i < len(candidate_features):
    var_actual = candidate_features[i]
    vars_sin_actual = candidate_features[:i] + candidate_features[i+1:]

    auc_tmp = mean_auc(X_traine2[vars_sin_actual], y_traine)

    # Decisión: feature_to_remove si mejora o se mantiene el AUC
    if auc_tmp >= best_auc:
        eliminado = True
        candidate_features = vars_sin_actual
        best_auc = auc_tmp
        # no avanzar i, porque los índices cambiaron
    else:
        eliminado = False
        i += 1  # pasar a la siguiente variable

    registro.append({
        'variable': var_actual,
        'auc_nuevo': auc_tmp,
        'eliminado': eliminado,
        'auc_base': best_auc
    })

# Convertir results a DataFrame
feature_elimination_results = pd.DataFrame(registro)



In [ ]:
len(candidate_features)

In [ ]:
X_traine2 = X_traine[candidate_features]
X_train2 = X_train[candidate_features]
X_test2 = X_test[candidate_features]
X_val2 = X_val[candidate_features]
X_cruz2 = X_cruz[candidate_features]
X_traine2

In [ ]:

# X: feature matrix
# y: binary target vector

candidate_features = list(X_traine2.columns)

# Function to compute mean AUC using cross-validation
def mean_auc(X_subset, y, cv_folds=5):
    model = LogisticRegression(C= 0.1, solver= 'liblinear')
    aucs = cross_val_score(model, X_subset, y, cv=cv_folds, scoring='roc_auc')
    return aucs.mean()

# AUC inicial con todas las candidate_features
best_auc = mean_auc(X_traine2[candidate_features], y_traine)

# Save figure el registro del proceso
registro = []

# Iteración secuencial
i = 0
while i < len(candidate_features):
    var_actual = candidate_features[i]
    vars_sin_actual = candidate_features[:i] + candidate_features[i+1:]

    auc_tmp = mean_auc(X_traine2[vars_sin_actual], y_traine)

    # Decisión: feature_to_remove si mejora o se mantiene el AUC
    if auc_tmp >= best_auc:
        eliminado = True
        candidate_features = vars_sin_actual
        best_auc = auc_tmp
        # no avanzar i, porque los índices cambiaron
    else:
        eliminado = False
        i += 1  # pasar a la siguiente variable

    registro.append({
        'variable': var_actual,
        'auc_nuevo': auc_tmp,
        'eliminado': eliminado,
        'auc_base': best_auc
    })

# Convertir results a DataFrame
feature_elimination_results = pd.DataFrame(registro)



In [ ]:
len(candidate_features)

In [ ]:
X_traine2 = X_traine[candidate_features]
X_train2 = X_train[candidate_features]
X_test2 = X_test[candidate_features]
X_val2 = X_val[candidate_features]
X_cruz2 = X_cruz[candidate_features]
X_traine2

In [ ]:
candidate_features

## 12. Final model training and evaluation

In [ ]:
final_model =LogisticRegression(C= 0.1, solver= 'liblinear')
#final_model=GradientBoostingClassifier( n_estimators= 100, criterion="friedman_mse")
final_model.fit(X_traine2, y_traine)
y_pred =final_model.predict(X_test2)
print(classification_report(y_test, y_pred))
cm =confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No response", "Response"])
disp.plot(cmap=plt.cm.Blues)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
# Estimate probabilities for the positive class
y_probabilities = final_model.predict_proba(X_test2)[:, 1]

# Compute AUC
auc = roc_auc_score(y_test, y_probabilities)
print(f"AUC: {auc:.4f}")

# Graficar curva ROC
fpr, tpr, _ = roc_curve(y_test, y_probabilities)
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")  # Línea de azar
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression Model")
plt.legend()
plt.savefig("roc_lr.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Predicted probabilities
y_probabilities = final_model.predict_proba(X_test2)[:, 1]

# AUC
auc = roc_auc_score(y_test, y_probabilities)

# ROC
fpr, tpr, _ = roc_curve(y_test, y_probabilities)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
plt.plot([0,1], [0,1], '--', color='gray')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")

# GUARDAR ANTES DE SHOW
plt.savefig("roc_lr.png", dpi=600, bbox_inches="tight")
plt.savefig("roc_lr.pdf", bbox_inches="tight")

plt.show()
plt.close()

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix, accuracy_score

# Class predictions
y_pred_test = final_model.predict(X_test2)

# Métricas individuales
accuracy = accuracy_score(y_test, y_pred_test)
recall = recall_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test)
f1 = f1_score(y_test, y_pred_test)

cm = confusion_matrix(y_test, y_pred_test)
if cm.shape == (2, 2):
    TN = cm[0, 0]
    FP = cm[0, 1]
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
else:
    specificity = float('nan')

print(f"Accuracy:     {accuracy:.2f}")
print(f"Recall:       {recall:.2f}")
print(f"Specificity:  {specificity:.2f}")
print(f"F1-score:     {f1:.2f}")
print(f"Precision:    {precision:.2f}")

In [ ]:
y_pred =final_model.predict(X_val2)
print(classification_report(y_val, y_pred))
cm =confusion_matrix(y_val, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No response", "Response"])
disp.plot(cmap=plt.cm.Blues)

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
# Estimate probabilities for the positive class
y_probabilities = final_model.predict_proba(X_val2)[:, 1]

# Compute AUC
auc = roc_auc_score(y_val, y_probabilities)
print(f"AUC: {auc:.4f}")

# Graficar curva ROC
fpr, tpr, _ = roc_curve(y_val, y_probabilities)
plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")  # Línea de azar
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression Model")
plt.legend()
plt.savefig("roc_lrval.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Predicted probabilities
y_probabilities = final_model.predict_proba(X_val2)[:, 1]

# AUC
auc = roc_auc_score(y_val, y_probabilities)

# ROC
fpr, tpr, _ = roc_curve(y_val, y_probabilities)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, lw=2, label=f"AUC = {auc:.3f}")
plt.plot([0,1], [0,1], '--', color='gray')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")

# GUARDAR ANTES DE SHOW
plt.savefig("roc_lr.png", dpi=600, bbox_inches="tight")
plt.savefig("roc_lr.pdf", bbox_inches="tight")

plt.show()
plt.close()

In [ ]:
from sklearn.metrics import recall_score, precision_score, f1_score, confusion_matrix, accuracy_score

# Class predictions
y_pred_test = final_model.predict(X_val2)

# Métricas individuales
accuracy = accuracy_score(y_val, y_pred_test)
recall = recall_score(y_val, y_pred_test)
precision = precision_score(y_val, y_pred_test)
f1 = f1_score(y_val, y_pred_test)

cm = confusion_matrix(y_val, y_pred_test)
if cm.shape == (2, 2):
    TN = cm[0, 0]
    FP = cm[0, 1]
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
else:
    specificity = float('nan')

print(f"Accuracy:     {accuracy:.2f}")
print(f"Recall:       {recall:.2f}")
print(f"Specificity:  {specificity:.2f}")
print(f"F1-score:     {f1:.2f}")
print(f"Precision:    {precision:.2f}")

In [ ]:
from sklearn.metrics import roc_auc_score
auc_scores = final_model.predict_proba(X_test2)[:, 1]
auc = roc_auc_score(y_test, auc_scores)
auc

In [ ]:
from sklearn.metrics import roc_auc_score
auc_scores = final_model.predict_proba(X_val2)[:, 1]
auc = roc_auc_score(y_val, auc_scores)
auc

In [ ]:
import joblib

model_output_path = '/content/drive/MyDrive/Colab Notebooks/AUNA/modelo_logistico1.pkl'
joblib.dump(final_model, model_output_path)

In [ ]:
final_gene_list =X_traine2.columns.to_numpy()
final_gene_list

In [ ]:
df3 =expression_matrix[final_gene_list]
df3

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, confusion_matrix,
    accuracy_score, recall_score
)

# ============================================================
# Threshold calibration and metrics on the test set
# ============================================================

def compute_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return tn / (tn + fp) if (tn + fp) > 0 else np.nan

def metrics_at_threshold(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    acc = accuracy_score(y_true, y_pred)
    sens = recall_score(y_true, y_pred, pos_label=1)  # sensibilidad
    spec = compute_specificity(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {
        "thr": float(thr),
        "accuracy": float(acc),
        "sensibilidad": float(sens),
        "especificidad": float(spec),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)
    }

def threshold_youden(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    youden = tpr - fpr  # = sens + espec - 1
    k = int(np.argmax(youden))
    return float(thr[k]), float(youden[k]), float(tpr[k]), float(1 - fpr[k])

def threshold_max_recall_min_spec(y_true, y_prob, min_spec=0.75):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    spec = 1 - fpr
    valid = spec >= min_spec
    if not np.any(valid):
        # No existe threshold que cumpla la restricción
        return np.nan, None

    # Entre los válidos, escoger el de mayor recall (tpr).
    # Si hay empates, escoger el de mayor especificidad.
    idxs = np.where(valid)[0]
    best = idxs[np.lexsort((-spec[idxs], -tpr[idxs]))][0]
    return float(thr[best]), {"recall": float(tpr[best]), "spec": float(spec[best])}

def plot_confusion(y_true, y_prob, thr, title):
    y_pred = (y_prob >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])

    fig, ax = plt.subplots(figsize=(5,4))

    im = ax.imshow(cm, cmap="Blues")

    ax.set_xlabel("Prediction")
    ax.set_ylabel("Real")

    ax.set_xticks([0,1])
    ax.set_yticks([0,1])

    ax.set_xticklabels(["No response (0)", "Response (1)"])
    ax.set_yticklabels(["No response (0)", "Response (1)"])

    # números dentro de la matriz
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=12)

    # título de la figura (no del eje)
    fig.suptitle(title, fontsize=12)

    # colorbar bien posicionada
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout(rect=[0,0,1,0.92])  # deja espacio para el título
    plt.show()
# ============================================================
# 1) Asegurar que el modelo final esté entrenado
# ============================================================
p_test = final_model.predict_proba(X_test2)[:, 1]

# ============================================================
# 2) Umbrales
# ============================================================
thr_05 = 0.5

thr_y, youdenJ, youden_sens, youden_spec = threshold_youden(y_test, p_test)

thr_r75, info_r75 = threshold_max_recall_min_spec(y_test, p_test, min_spec=0.75)

print("[Umbrales en TEST]")
print(f"thr_0.5 = {thr_05:.3f}")
print(f"thr_youden = {thr_y:.6f} | J={youdenJ:.3f} | sens={youden_sens:.3f} | spec={youden_spec:.3f}")
if np.isnan(thr_r75):
    print("thr_max_recall_spec0.75 = NO EXISTE threshold con especificidad >= 0.75 en TEST")
else:
    print(f"thr_max_recall_spec0.75 = {thr_r75:.6f} | sens={info_r75['recall']:.3f} | spec={info_r75['spec']:.3f}")

# ============================================================
# 3) Métricas en TEST para cada threshold
# ============================================================
rows = []
rows.append({"threshold_name": "thr_0.5", **metrics_at_threshold(y_test, p_test, thr_05)})
rows.append({"threshold_name": "thr_youden", **metrics_at_threshold(y_test, p_test, thr_y)})

if not np.isnan(thr_r75):
    rows.append({"threshold_name": "thr_max_recall_spec0.75", **metrics_at_threshold(y_test, p_test, thr_r75)})

df_thr = pd.DataFrame(rows)[[
    "threshold_name", "thr", "accuracy", "sensibilidad", "especificidad", "TN", "FP", "FN", "TP"
]]
display(df_thr)

# ============================================================
# 4) Matrices de confusión (una figura por threshold)
# ============================================================
plot_confusion(y_test, p_test, thr_05, "TEST | Matriz de confusión | Umbral 0.5")
plot_confusion(y_test, p_test, thr_y,  f"TEST | Matriz de confusión | Youden (thr={thr_y:.3f})")

if not np.isnan(thr_r75):
    plot_confusion(y_test, p_test, thr_r75, f"TEST | Matriz de confusión | Max recall con spec≥0.75 (thr={thr_r75:.3f})")

In [ ]:
# ============================================================
# Apply thresholds calibrated on TEST to the validation set
# ============================================================

# Predicted probabilities en VAL
p_val = final_model.predict_proba(X_val2)[:, 1]

# Métricas en VAL con thresholds de TEST
rows_val = []
rows_val.append({"threshold_name": "thr_0.5 (from TEST)", **metrics_at_threshold(y_val, p_val, thr_05)})
rows_val.append({"threshold_name": "thr_youden (from TEST)", **metrics_at_threshold(y_val, p_val, thr_y)})

if not np.isnan(thr_r75):
    rows_val.append({
        "threshold_name": "thr_max_recall_spec0.75 (from TEST)",
        **metrics_at_threshold(y_val, p_val, thr_r75)
    })

df_thr_val = pd.DataFrame(rows_val)[[
    "threshold_name", "thr", "accuracy", "sensibilidad", "especificidad", "TN", "FP", "FN", "TP"
]]
display(df_thr_val)

# Matrices de confusión en VAL (threshold fijo desde TEST)
plot_confusion(y_val, p_val, thr_05, "VAL | Matriz de confusión | Umbral 0.5 (calibrado en TEST)")
plot_confusion(y_val, p_val, thr_y,  f"VAL | Matriz de confusión | Youden de TEST (thr={thr_y:.3f})")

if not np.isnan(thr_r75):
    plot_confusion(
        y_val, p_val, thr_r75,
        f"VAL | Matriz de confusión | Max recall con spec≥0.75 de TEST (thr={thr_r75:.3f})"
    )
else:
    print("[INFO] No se grafica el threshold max_recall_spec0.75 en VAL porque no existió en TEST.")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Output directory
outdir = "figures_confusion_matrices"
os.makedirs(outdir, exist_ok=True)

# Predicted probabilities en VAL
p_val = final_model.predict_proba(X_val2)[:, 1]

# Función mejorada para guardar matriz
def save_confusion_matrix(y_true, y_prob, thr, title, filename):
    y_pred = (y_prob >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    fig, ax = plt.subplots(figsize=(5, 5))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Non-responder", "Responser"]
    )
    disp.plot(cmap="Blues", values_format="d", ax=ax, colorbar=False)

    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    plt.tight_layout()

    png_path = os.path.join(outdir, f"{filename}.png")
    pdf_path = os.path.join(outdir, f"{filename}.pdf")

    plt.savefig(png_path, dpi=600, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.show()
    plt.close()

    return cm, png_path, pdf_path


# Métricas en VAL con thresholds calibrados en TEST
rows_val = []

rows_val.append({
    "threshold_name": "0.5",
    **metrics_at_threshold(y_val, p_val, thr_05)
})

rows_val.append({
    "threshold_name": "Youden index",
    **metrics_at_threshold(y_val, p_val, thr_y)
})

if not np.isnan(thr_r75):
    rows_val.append({
        "threshold_name": "Max recall with specificity ≥ 0.75",
        **metrics_at_threshold(y_val, p_val, thr_r75)
    })

df_thr_val = pd.DataFrame(rows_val)[[
    "threshold_name", "thr", "accuracy", "sensibilidad",
    "especificidad", "TN", "FP", "FN", "TP"
]]

display(df_thr_val)

# Save figure tabla de métricas
df_thr_val.to_csv(
    os.path.join(outdir, "test_threshold_metrics.csv"),
    index=False
)

df_thr_val.to_excel(
    os.path.join(outdir, "test_threshold_metrics.xlsx"),
    index=False
)


# Matriz threshold 0.5
save_confusion_matrix(
    y_val, p_val, thr_05,
    "test cohort | Threshold = 0.5",
    "confusion_matrix_validation_thr_05"
)

# Matriz Youden
save_confusion_matrix(
    y_val, p_val, thr_y,
    f"test cohort | Youden threshold = {thr_y:.3f}",
    "confusion_matrix_validation_youden"
)

# Matriz especificidad ≥ 0.75
if not np.isnan(thr_r75):
    save_confusion_matrix(
        y_val, p_val, thr_r75,
        f"test cohort | Max recall, specificity ≥ 0.75\nThreshold = {thr_r75:.3f}",
        "confusion_matrix_validation_spec075"
    )
else:
    print("[INFO] No se grafica el threshold max_recall_spec0.75 porque no existió en TEST.")

print(f"Archivos guardados en: {outdir}")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

# ============================================================
# Functions
# ============================================================

def compute_metrics(y_true, y_prob, threshold=0.5):
    """
    Calculate classification metrics.
    """

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_prob)

    accuracy = accuracy_score(y_true, y_pred)

    sensitivity = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0,1]
    ).ravel()

    specificity = tn / (tn + fp)

    return {
        "AUC": auc,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1-score": f1
    }


# ============================================================
# Bootstrap
# ============================================================

def bootstrap_ci_metrics(
    y_true,
    y_prob,
    threshold=0.5,
    n_bootstrap=2000,
    random_state=42
):

    rng = np.random.default_rng(random_state)

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    n = len(y_true)

    # Point estimates from the original dataset
    original = compute_metrics(
        y_true,
        y_prob,
        threshold
    )

    boot_metrics = []

    for i in range(n_bootstrap):

        idx = rng.choice(
            n,
            size=n,
            replace=True
        )

        yt = y_true[idx]
        yp = y_prob[idx]

        # Skip bootstrap samples containing only one class
        if len(np.unique(yt)) < 2:
            continue

        boot_metrics.append(
            compute_metrics(
                yt,
                yp,
                threshold
            )
        )

    boot_df = pd.DataFrame(boot_metrics)

    summary = []

    for metric in original.keys():

        lower = np.percentile(
            boot_df[metric],
            2.5
        )

        upper = np.percentile(
            boot_df[metric],
            97.5
        )

        summary.append({
            "Metric": metric,
            "Estimate": original[metric],
            "CI Lower": lower,
            "CI Upper": upper,
            "Estimate (95% CI)":
                f"{original[metric]:.3f} "
                f"({lower:.3f}-{upper:.3f})"
        })

    summary = pd.DataFrame(summary)

    return summary


# ============================================================
# TEST SET
# ============================================================

p_test = final_model.predict_proba(X_test2)[:,1]

summary_test = bootstrap_ci_metrics(
    y_true = y_test,
    y_prob=p_test,
    threshold=0.5,
    n_bootstrap=2000,
    random_state=42
)

summary_test["Dataset"] = "Test"


# ============================================================
# VALIDATION SET
# ============================================================

p_val = final_model.predict_proba(X_val2)[:,1]

summary_val = bootstrap_ci_metrics(
    y_true = y_val,
    y_prob=p_val,
    threshold=0.5,
    n_bootstrap=2000,
    random_state=43
)

summary_val["Dataset"] = "Validation"


# ============================================================
# Merge results
# ============================================================

summary_all = pd.concat(
    [
        summary_test,
        summary_val
    ],
    ignore_index=True
)

summary_all = summary_all[
    [
        "Dataset",
        "Metric",
        "Estimate",
        "CI Lower",
        "CI Upper",
        "Estimate (95% CI)"
    ]
]

print(summary_all)

display(summary_all)


# ============================================================
# Save results
# ============================================================

summary_all.to_csv(
    "Bootstrap_95CI_Test_Validation.csv",
    index=False
)

summary_all.to_excel(
    "Bootstrap_95CI_Test_Validation.xlsx",
    index=False
)

print("\nResults saved successfully.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

# ==========================
# Publication-quality settings
# ==========================
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 14,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12
})

# ==========================
# Predicted probabilities
# ==========================
y_test_prob = final_model.predict_proba(X_test2)[:, 1]

# ==========================
# Brier score
# ==========================
brier = brier_score_loss(y_test, y_test_prob)
print(f"Brier score: {brier:.3f}")

# ==========================
# Calibration curve
# ==========================
prob_true, prob_pred = calibration_curve(
    y_test,
    y_test_prob,
    n_bins=5,
    strategy="quantile"
)

# ==========================
# Plot
# ==========================
fig, ax = plt.subplots(figsize=(5.5, 5.5))

ax.plot(
    prob_pred,
    prob_true,
    marker="o",
    linewidth=2,
    markersize=7,
    label=f"Model (Brier score = {brier:.3f})"
)

ax.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    color="gray",
    label="Perfect calibration"
)

ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Response Rate")
ax.set_title("Calibration Curve")

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.legend(frameon=False, loc="upper left")

ax.grid(
    linestyle="--",
    linewidth=0.6,
    alpha=0.4
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()

# ==========================
# Save figure
# ==========================
plt.savefig(
    "Calibration_Curve_Test.tiff",
    format="tiff",
    dpi=600,
    bbox_inches="tight",
    pil_kwargs={"compression": "raw"}
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

# ============================================================
# Data used to train the final model
# ============================================================

X_train_or = X_traine2.copy()
y_train_or = np.asarray(y_traine)

selected_genes = X_train_or.columns

# ============================================================
# Fit the final model on the original training data
# ============================================================

final_model = LogisticRegression(
    C=0.1,
    solver='liblinear',
    max_iter=1000
)

final_model.fit(X_train_or, y_train_or)

# Coefficients from the final model
coef_original = final_model.coef_[0]

# Odds ratios from the final model
or_original = np.exp(coef_original)

# ============================================================
# Bootstrap coefficients
# ============================================================

n_bootstrap = 2000
random_state = 42
rng = np.random.default_rng(random_state)

boot_coefs = []

n = X_train_or.shape[0]

for i in range(n_bootstrap):

    # Resample patients with replacement
    idx = rng.choice(
        np.arange(n),
        size=n,
        replace=True
    )

    X_boot = X_train_or.iloc[idx]
    y_boot = y_train_or[idx]

    # Skip bootstrap samples with only one class
    if len(np.unique(y_boot)) < 2:
        continue

    model_boot = LogisticRegression(
        C=0.1,
        solver='liblinear',
        max_iter=1000
    )

    model_boot.fit(X_boot, y_boot)

    boot_coefs.append(model_boot.coef_[0])

boot_coefs = np.array(boot_coefs)

# Convert bootstrap coefficients to OR
boot_or = np.exp(boot_coefs)

# ============================================================
# 95% confidence intervals
# ============================================================

or_ci_lower = np.percentile(boot_or, 2.5, axis=0)
or_ci_upper = np.percentile(boot_or, 97.5, axis=0)

# ============================================================
# Create results table
# ============================================================

or_table = pd.DataFrame({
    "Gene": selected_genes,
    "Coefficient": coef_original,
    "OR": or_original,
    "CI_lower": or_ci_lower,
    "CI_upper": or_ci_upper
})

# Direction of association
or_table["Direction"] = np.where(
    or_table["OR"] > 1,
    "Associated with response",
    "Associated with non-response"
)

# Formatted column for manuscript
or_table["OR (95% CI)"] = or_table.apply(
    lambda x: f"{x['OR']:.2f} ({x['CI_lower']:.2f}-{x['CI_upper']:.2f})",
    axis=1
)

# Sort by absolute coefficient magnitude
or_table["abs_coefficient"] = or_table["Coefficient"].abs()

or_table = or_table.sort_values(
    "abs_coefficient",
    ascending=False
)

# Final table
or_table_final = or_table[
    [
        "Gene",
        "Coefficient",
        "OR",
        "CI_lower",
        "CI_upper",
        "OR (95% CI)",
        "Direction"
    ]
]

display(or_table_final)

# ============================================================
# Save table
# ============================================================

or_table_final.to_csv(
    "Logistic_Regression_OR_95CI_bootstrap.csv",
    index=False
)

or_table_final.to_excel(
    "Logistic_Regression_OR_95CI_bootstrap.xlsx",
    index=False
)

print("OR table saved successfully.")

In [ ]:
# Starting from the final odds-ratio table

or_table_nr = or_table_final.copy()

# Odds ratios for non-response
or_table_nr["OR_non_response"] = 1 / or_table_nr["OR"]
or_table_nr["CI_lower_non_response"] = 1 / or_table_nr["CI_upper"]
or_table_nr["CI_upper_non_response"] = 1 / or_table_nr["CI_lower"]

# Interpret direction as increased risk of non-response
or_table_nr["Direction_non_response"] = np.where(
    or_table_nr["OR_non_response"] > 1,
    "Associated with non-response",
    "Associated with response"
)

# Manuscript-ready formatting
or_table_nr["OR for non-response (95% CI)"] = or_table_nr.apply(
    lambda x: f"{x['OR_non_response']:.2f} "
              f"({x['CI_lower_non_response']:.2f}-{x['CI_upper_non_response']:.2f})",
    axis=1
)

# Tabla final
or_table_nr_final = or_table_nr[
    [
        "Gene",
        "Coefficient",
        "OR_non_response",
        "CI_lower_non_response",
        "CI_upper_non_response",
        "OR for non-response (95% CI)",
        "Direction_non_response"
    ]
]

display(or_table_nr_final)

# Save figure
or_table_nr_final.to_csv(
    "Logistic_Regression_OR_for_NonResponse_95CI.csv",
    index=False
)

or_table_nr_final.to_excel(
    "Logistic_Regression_OR_for_NonResponse_95CI.xlsx",
    index=False
)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    confusion_matrix
)

from imblearn.over_sampling import SMOTE


# ============================================================
# 1. Get cancer type from clinical metadata using run_accession
# ============================================================

def get_cancer_type_for_matrix(X, clinical_metadata):
    clinical = clinical_metadata.set_index("run_accession")
    return clinical.loc[X.index, "Tipo"]


cancer_type_train = get_cancer_type_for_matrix(X_train2, clinical_metadata)
cancer_type_validation   = get_cancer_type_for_matrix(X_val2, clinical_metadata)
cancer_type_test  = get_cancer_type_for_matrix(X_test2, clinical_metadata)


# ============================================================
# 2. Training pool: TRAIN + VALIDATION only
# ============================================================

X_train_pool = pd.concat([X_train2, X_val2], axis=0)

y_train_pool = pd.concat([
    pd.Series(y_train, index=X_train2.index),
    pd.Series(y_val, index=X_val2.index)
], axis=0)

tipo_train_pool = pd.concat([cancer_type_train, cancer_type_validation], axis=0)


# ============================================================
# 3. Evaluation pool: TRAIN + VALIDATION + TEST
#    Only to evaluate all samples of the excluded cancer type
# ============================================================

X_eval_pool = pd.concat([X_train2, X_val2, X_test2], axis=0)

y_eval_pool = pd.concat([
    pd.Series(y_train, index=X_train2.index),
    pd.Series(y_val, index=X_val2.index),
    pd.Series(y_test, index=X_test2.index)
], axis=0)

tipo_eval_pool = pd.concat([cancer_type_train, cancer_type_validation, cancer_type_test], axis=0)


# Checks
assert all(X_train_pool.index == y_train_pool.index)
assert all(X_train_pool.index == tipo_train_pool.index)

assert all(X_eval_pool.index == y_eval_pool.index)
assert all(X_eval_pool.index == tipo_eval_pool.index)

print("Training pool:", X_train_pool.shape)
print(tipo_train_pool.value_counts())

print("\nEvaluation pool:", X_eval_pool.shape)
print(tipo_eval_pool.value_counts())


# ============================================================
# 4. Metric functions
# ============================================================

def compute_metrics(y_true, y_prob, threshold=0.5):

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan

    accuracy = accuracy_score(y_true, y_pred)

    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )

    sensitivity = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "AUC": auc,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1-score": f1,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }


def bootstrap_ci_metrics(
    y_true,
    y_prob,
    threshold=0.5,
    n_bootstrap=2000,
    seed=42
):

    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    n = len(y_true)
    rng = np.random.default_rng(seed)

    boot_results = []

    for _ in range(n_bootstrap):

        idx = rng.choice(
            np.arange(n),
            size=n,
            replace=True
        )

        yt = y_true[idx]
        yp = y_prob[idx]

        if len(np.unique(yt)) < 2:
            continue

        boot_results.append(
            compute_metrics(
                yt,
                yp,
                threshold=threshold
            )
        )

    boot_df = pd.DataFrame(boot_results)

    ci = {}

    for metric in [
        "AUC",
        "Accuracy",
        "Balanced Accuracy",
        "Sensitivity",
        "Specificity",
        "Precision",
        "F1-score"
    ]:
        ci[f"{metric}_CI_lower"] = np.nanpercentile(boot_df[metric], 2.5)
        ci[f"{metric}_CI_upper"] = np.nanpercentile(boot_df[metric], 97.5)

    return ci


# ============================================================
# 5. Leave-one-cancer-type generalization analysis
# ============================================================

results = []

for cancer_type in tipo_eval_pool.unique():

    print(f"\nProcessing excluded cancer type: {cancer_type}")

    # Train only on TRAIN + VAL, excluding this cancer type
    train_mask = tipo_train_pool != cancer_type

    X_train_cv = X_train_pool.loc[train_mask]
    y_train_cv = y_train_pool.loc[train_mask]

    # Evaluate on ALL available samples of the excluded cancer type
    test_mask = tipo_eval_pool == cancer_type

    X_test_cv = X_eval_pool.loc[test_mask]
    y_test_cv = y_eval_pool.loc[test_mask]

    if len(np.unique(y_train_cv)) < 2:
        print(f"Skipping {cancer_type}: training set has only one class.")
        continue

    # SMOTE only on training raw_counts
    class_counts = pd.Series(y_train_cv).value_counts()
    minority_count = class_counts.min()

    if minority_count > 1:
        smote = SMOTE(
            random_state=42,
            k_neighbors=min(5, minority_count - 1)
        )

        X_train_resampled, y_train_resampled = smote.fit_resample(
            X_train_cv,
            y_train_cv
        )
    else:
        X_train_resampled = X_train_cv
        y_train_resampled = y_train_cv

    # Train Logistic Regression
    model = LogisticRegression(
        C=0.1,
        solver="liblinear",
        max_iter=1000
    )

    model.fit(
        X_train_resampled,
        y_train_resampled
    )

    # Probability of response
    y_prob_cv = model.predict_proba(X_test_cv)[:, 1]

    # Point estimates
    metrics = compute_metrics(
        y_true = y_test_cv,
        y_prob = y_prob_cv,
        threshold=0.5
    )

    # Bootstrap 95% CI
    ci_metrics = bootstrap_ci_metrics(
        y_true = y_test_cv,
        y_prob = y_prob_cv,
        threshold=0.5,
        n_bootstrap=2000,
        seed=42
    )

    results.append({
        "Cancer type": cancer_type,
        "n": len(y_test_cv),
        "Responsers": int(np.sum(y_test_cv == 1)),
        "Non-responders": int(np.sum(y_test_cv == 0)),
        **metrics,
        **ci_metrics
    })


# ============================================================
# 6. Results table for manuscript
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="AUC",
    ascending=False,
    na_position="last"
)

for metric in [
    "AUC",
    "Balanced Accuracy",
    "Sensitivity",
    "Specificity"
]:
    results_df[f"{metric} (95% CI)"] = results_df.apply(
        lambda x: (
            f"{x[metric]:.3f} "
            f"({x[f'{metric}_CI_lower']:.3f}-{x[f'{metric}_CI_upper']:.3f})"
        ),
        axis=1
    )

main_table = results_df[
    [
        "Cancer type",
        "n",
        "Responsers",
        "Non-responders",
        "AUC (95% CI)",
        "Balanced Accuracy (95% CI)",
        "Sensitivity (95% CI)",
        "Specificity (95% CI)"
    ]
]

print("\nMain table for manuscript\n")
print(main_table)

display(main_table)


# ============================================================
# 7. Full supplementary table
# ============================================================

for metric in [
    "Accuracy",
    "Precision",
    "F1-score"
]:
    results_df[f"{metric} (95% CI)"] = results_df.apply(
        lambda x: (
            f"{x[metric]:.3f} "
            f"({x[f'{metric}_CI_lower']:.3f}-{x[f'{metric}_CI_upper']:.3f})"
        ),
        axis=1
    )

supplementary_table = results_df[
    [
        "Cancer type",
        "n",
        "Responsers",
        "Non-responders",
        "AUC (95% CI)",
        "Accuracy (95% CI)",
        "Balanced Accuracy (95% CI)",
        "Sensitivity (95% CI)",
        "Specificity (95% CI)",
        "Precision (95% CI)",
        "F1-score (95% CI)",
        "TN",
        "FP",
        "FN",
        "TP"
    ]
]

print("\nSupplementary table\n")
print(supplementary_table)

display(supplementary_table)


# ============================================================
# 8. Save tables
# ============================================================

main_table.to_csv(
    "Leave_One_Cancer_Type_Main_Table.csv",
    index=False
)

main_table.to_excel(
    "Leave_One_Cancer_Type_Main_Table.xlsx",
    index=False
)

supplementary_table.to_csv(
    "Leave_One_Cancer_Type_Supplementary_Table.csv",
    index=False
)

supplementary_table.to_excel(
    "Leave_One_Cancer_Type_Supplementary_Table.xlsx",
    index=False
)

print("\nAnalysis completed and tables saved.")

## GenCell

> The notebook is intentionally truncated at the **GenCell** section, as requested.
